# M4 · Set sintético de imágenes 

## 1. Clonar repositorio

In [ ]:
# Rama que contiene M4/eval_visual (debe estar subida a GitHub)
RAMA = "LuisM4"

!git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo

In [ ]:
%cd /content/repo
!git pull
%cd M4/eval_visual
%ls

## 2. Instalar dependencias

In [ ]:
!pip install -r requirements.txt -q

## 3. Montar Drive y definir `PROJECT_ROOT`

Las imágenes se guardan en `<PROJECT_ROOT>/M4/data/imagenes/` (fuera de git). El gold set se lee del repo clonado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# .env en la raiz del repo: load_dotenv() lo encuentra al subir desde M4/eval_visual
with open("/content/repo/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
print(".env generado (no versionado, solo vive en este runtime)")

## 4. Revisar con una muestra 

Genera solo 3 documentos para comprobar que todo funciona antes de la corrida completa.

In [ ]:
!python generar_set.py --limite 3

## 5. Generar el set completo

28 niveles: 3 base con los 59 documentos (`limpio`, `foto_realista`, `manuscrito_foto`) y 25 barridos de severidad sobre un subconjunto fijo de 20 documentos. Ver el README para las cifras y el tiempo medidos.

Para generar solo algunos niveles: `!python generar_set.py --niveles limpio foto_realista manuscrito_foto`

In [ ]:
!python generar_set.py

## 6. Verificar el resultado

In [ ]:
import json
from IPython.display import display
from PIL import Image

IMAGENES = PROJECT_ROOT / "M4/data/imagenes"
with open(IMAGENES / "manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Documentos: {manifest['n_documentos']} | Imagenes: {manifest['n_imagenes']}")
print(f"{'nivel':28s} docs  pags  %palabras_ocultas  entidades no visibles")
for nivel, r in manifest["niveles"].items():
    e = r["entidades"]
    print(f"{nivel:28s} {r['documentos']:4d} {r['paginas']:5d} {r['pct_palabras_ocultas']:12.2f}"
          f"      {e['parcial'] + e['oculta']} de {e['total']}")

# Primera pagina del primer documento del subconjunto de barrido, en cada nivel
doc_id = manifest["subconjuntos"]["barrido"][0]
doc = next(d for d in manifest["documentos"] if d["doc_id"] == doc_id)
for nivel, info in doc["niveles"].items():
    pagina = info["paginas"][0]
    aplicados = [e["tipo"] for e in pagina["efectos"] if e["aplicado"]]
    print(nivel, "| paginas:", info["n_paginas"], "| efectos:", aplicados)
    img = Image.open(IMAGENES / pagina["archivo"])
    img.thumbnail((600, 850))
    display(img)